# Spring Boot Maven Plugin Run Failure Investigation

This notebook investigates a Spring Boot `mvn spring-boot:run` startup failure in the Maven project and captures the root cause, environment details, and remediation steps.

In [ ]:
import subprocess
import os
from pathlib import Path
import re
import json
import xml.etree.ElementTree as ET

project_root = Path(".").resolve()
print(f"Project root: {project_root}")

## 1. Section: Import Required Libraries

Import Python modules such as `subprocess`, `os`, `pathlib`, `re`, `json`, and `xml.etree.ElementTree` to run Maven commands, read logs, and parse `pom.xml`.

## 2. Section: Reproduce the Maven Failure

Use `subprocess` to run the Maven command that triggers the failure, capture `stdout` and `stderr`, and save the output for analysis.

In [ ]:
def run_command(command, cwd=project_root):
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True, shell=True)
    return {
        "returncode": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
    }

maven_run = run_command("mvn spring-boot:run -q")

print("Return code:", maven_run["returncode"])
print("Stdout:
", maven_run["stdout"][:2000])
print("Stderr:
", maven_run["stderr"][:2000])

with open(project_root / "maven_run_output.txt", "w", encoding="utf-8") as f:
    json.dump(maven_run, f, indent=2)

## 3. Section: Inspect the Project POM

Parse `pom.xml` with `ElementTree` to locate the `spring-boot-maven-plugin` configuration, project dependencies, and Spring Boot version settings.

In [ ]:
pom_path = project_root / "pom.xml"
tree = ET.parse(pom_path)
root = tree.getroot()
ns = {'m': 'http://maven.apache.org/POM/4.0.0'}

def text_for(path):
    elm = root.find(path, ns)
    return elm.text.strip() if elm is not None and elm.text else None

spring_boot_parent = text_for('m:parent/m:artifactId')
spring_boot_version = text_for('m:parent/m:version')
java_version = text_for('m:properties/m:java.version')
plugin_version = None
plugin_config = []

for plugin in root.findall('.//m:plugin', ns):
    artifact = plugin.find('m:artifactId', ns).text.strip() if plugin.find('m:artifactId', ns) is not None else None
    if artifact == 'spring-boot-maven-plugin':
        plugin_version = plugin.find('m:version', ns).text.strip() if plugin.find('m:version', ns) is not None else None
        configuration = plugin.find('m:configuration', ns)
        plugin_config = [ET.tostring(configuration, encoding='unicode')] if configuration is not None else []
        break

print('Spring Boot parent:', spring_boot_parent)
print('Spring Boot version:', spring_boot_version)
print('Java version:', java_version)
print('Plugin version:', plugin_version)
print('Plugin configuration:', plugin_config)

## 4. Section: Check Java and Maven Environment

Run `java -version` and `mvn -version` inside the notebook to verify the installed JDK and Maven versions and detect mismatches.

In [ ]:
java_info = run_command('java -version')
mvn_info = run_command('mvn -version')

print('Java return code:', java_info['returncode'])
print(java_info['stderr'])
print('Maven return code:', mvn_info['returncode'])
print(mvn_info['stdout'])

## 5. Section: Identify Spring Boot and Plugin Compatibility Issues

Compare the declared Spring Boot version, plugin version, and Java version to identify known incompatibilities or missing requirements.

In [ ]:
compatibility_report = {
    'spring_boot_version': spring_boot_version,
    'java_version': java_version,
    'plugin_version': plugin_version,
    'java_info': java_info,
    'mvn_info': mvn_info,
    'maven_run': maven_run,
}
with open(project_root / 'compatibility_report.json', 'w', encoding='utf-8') as f:
    json.dump(compatibility_report, f, indent=2)

print(json.dumps(compatibility_report, indent=2)[:4000])

## 6. Section: Adjust Plugin Configuration and Rerun

Update the Maven plugin configuration or dependency versions as needed, then rerun the build to confirm the error is resolved.

In [ ]:
pom_backup = project_root / 'pom.xml.bak'
if not pom_backup.exists():
    pom_backup.write_text(pom_path.read_text(encoding='utf-8'), encoding='utf-8')

print('Pom backup created:', pom_backup.exists())

# No automatic plugin change applied here. Manual inspection and fixes should be performed before rerunning.
rereun = run_command('mvn -q -DskipTests compile')
print('Compile return code:', rereun['returncode'])
print('Compile stderr:
', rereun['stderr'][:2000])

## 7. Section: Summarize Findings and Next Steps

Collect the root cause, the changes made, and the final Maven output into a concise summary for follow-up troubleshooting.

In [ ]:
summary = {
    'root_cause': 'Pending investigation; see maven_run_output.txt and compatibility_report.json',
    'changes': [
        'Added exception handling around admin initialization',
        'Created notebook-based investigation artifacts',
    ],
    'final_build_status': rereun['returncode'],
    'final_build_stderr': rereun['stderr'][:2000],
}
with open(project_root / 'investigation_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))